# Feature Stacking - Phase 2

Building on rf_ecfp_tuned (Val F1: 0.7012), this notebook stacks multiple fingerprints using sklearn's `make_union`:
- ECFP (count)
- MACCS keys (166 bits, substructure-based)
- Atom Pair fingerprints
- Topological Torsion fingerprints

Following scikit-fingerprints examples: fingerprints can take SMILES directly.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import f1_score
from sklearn.pipeline import make_union
from skfp.fingerprints import (
    ECFPFingerprint,
    MACCSFingerprint,
    AtomPairFingerprint,
    TopologicalTorsionFingerprint,
)
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Load data
train_df = pd.read_parquet('../data/train.parquet')
val_df = pd.read_parquet('../data/val.parquet')
test_df = pd.read_parquet('../chebi_dataset_test_empty.parquet')
label_cols = [c for c in train_df.columns if c.startswith('class_')]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, Classes: {len(label_cols)}")

Train: 26934, Val: 6734, Test: 11223, Classes: 500


## Create Feature Union

Using `make_union` from sklearn to concatenate fingerprints. Fingerprints take SMILES directly.

In [2]:
# Create feature union - fingerprints accept SMILES directly
fp_union = make_union(
    ECFPFingerprint(fp_size=2048, radius=2, count=True, n_jobs=-1),
    MACCSFingerprint(n_jobs=-1),
    AtomPairFingerprint(fp_size=2048, count=True, n_jobs=-1),
    TopologicalTorsionFingerprint(fp_size=2048, count=True, n_jobs=-1),
    n_jobs=-1,
)

print("Computing stacked fingerprints...")
X_train = fp_union.fit_transform(train_df['SMILES'].tolist())
X_val = fp_union.transform(val_df['SMILES'].tolist())
X_test = fp_union.transform(test_df['SMILES'].tolist())

y_train = train_df[label_cols].values
y_val = val_df[label_cols].values

print(f"Stacked features: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

Computing stacked fingerprints...


[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not removing hydrogen atom without neighbors
[14:40:29] WARNING: not r

Stacked features: X_train=(26934, 6310), X_val=(6734, 6310), X_test=(11223, 6310)


In [3]:
# Optional: filter zero-variance features
var_filter = VarianceThreshold()
X_train_filtered = var_filter.fit_transform(X_train)
X_val_filtered = var_filter.transform(X_val)
X_test_filtered = var_filter.transform(X_test)

print(f"After variance filter: {X_train.shape[1]} -> {X_train_filtered.shape[1]} features")

After variance filter: 6310 -> 6207 features


## Train Random Forest

In [4]:
# Train RF on filtered features
rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, class_weight='balanced', random_state=42)
rf.fit(X_train_filtered, y_train)

# Get probabilities
proba_list = rf.predict_proba(X_val_filtered)
y_val_proba = np.array([p[:, 1] if p.shape[1] == 2 else p[:, 0] for p in proba_list]).T

print(f"Val proba shape: {y_val_proba.shape}")

Val proba shape: (6734, 500)


In [5]:
# Baseline F1
y_val_pred_default = (y_val_proba >= 0.5).astype(int)
f1_default = np.mean([f1_score(y_val[:, i], y_val_pred_default[:, i], zero_division=0) for i in range(len(label_cols))])
print(f"Stacked Features Val Macro F1 (threshold=0.5): {f1_default:.4f}")

Stacked Features Val Macro F1 (threshold=0.5): 0.3391


## Threshold Tuning

In [6]:
def optimize_threshold(y_true, y_proba, thresholds=np.arange(0.1, 0.91, 0.05)):
    """Find optimal threshold for a single class."""
    best_thresh = 0.5
    best_f1 = 0
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    return best_thresh, best_f1

# Optimize thresholds
optimal_thresholds = []
per_class_f1 = []

for i in range(len(label_cols)):
    thresh, f1 = optimize_threshold(y_val[:, i], y_val_proba[:, i])
    optimal_thresholds.append(thresh)
    per_class_f1.append(f1)

optimal_thresholds = np.array(optimal_thresholds)
print(f"Threshold stats: mean={optimal_thresholds.mean():.3f}, min={optimal_thresholds.min():.3f}, max={optimal_thresholds.max():.3f}")
print(f"Stacked + Threshold Tuned Val Macro F1: {np.mean(per_class_f1):.4f}")

Threshold stats: mean=0.279, min=0.100, max=0.900
Stacked + Threshold Tuned Val Macro F1: 0.6489


## DAG Consistency

In [7]:
def parse_obo(filepath):
    """Parse OBO file to extract parent-child relationships."""
    parents = defaultdict(list)
    children = defaultdict(list)
    
    current_id = None
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('id: class_'):
                current_id = line.split('id: ')[1]
            elif line.startswith('is_a: class_') and current_id:
                parent_id = line.split('is_a: ')[1].split(' !')[0]
                parents[current_id].append(parent_id)
                children[parent_id].append(current_id)
    
    return dict(parents), dict(children)

parents_map, children_map = parse_obo('../data/chebi_classes.obo')
id_to_idx = {col: i for i, col in enumerate(label_cols)}
print(f"Parsed DAG: {len(parents_map)} nodes with parents, {len(children_map)} with children")

Parsed DAG: 499 nodes with parents, 312 with children


In [8]:
def enforce_dag_consistency(proba, parents_map, id_to_idx):
    """Enforce P(parent) >= max(P(children)) via bottom-up propagation."""
    proba = proba.copy()
    our_ids = set(id_to_idx.keys())
    
    for _ in range(50):
        changed = False
        for child_id, parent_ids in parents_map.items():
            if child_id not in our_ids:
                continue
            child_idx = id_to_idx[child_id]
            
            for parent_id in parent_ids:
                if parent_id not in our_ids:
                    continue
                parent_idx = id_to_idx[parent_id]
                mask = proba[:, parent_idx] < proba[:, child_idx]
                if mask.any():
                    proba[mask, parent_idx] = proba[mask, child_idx]
                    changed = True
        if not changed:
            break
    return proba

# Apply DAG consistency
y_val_proba_dag = enforce_dag_consistency(y_val_proba, parents_map, id_to_idx)

# Re-optimize thresholds
optimal_thresholds_dag = []
per_class_f1_dag = []

for i in range(len(label_cols)):
    thresh, f1 = optimize_threshold(y_val[:, i], y_val_proba_dag[:, i])
    optimal_thresholds_dag.append(thresh)
    per_class_f1_dag.append(f1)

optimal_thresholds_dag = np.array(optimal_thresholds_dag)
print(f"Stacked + DAG + Threshold Tuned Val Macro F1: {np.mean(per_class_f1_dag):.4f}")

Stacked + DAG + Threshold Tuned Val Macro F1: 0.6489


## Generate Test Submission

In [ ]:
# Retrain on full data
full_df = pd.concat([train_df, val_df], ignore_index=True)

print("Computing fingerprints for full data...")
X_full = fp_union.transform(full_df['SMILES'].tolist())
X_full_filtered = var_filter.transform(X_full)
y_full = full_df[label_cols].values

rf_full = RandomForestClassifier(n_estimators=50, n_jobs=-1, class_weight='balanced', random_state=42)
rf_full.fit(X_full_filtered, y_full)
print(f"Trained on full data: {X_full_filtered.shape}")

In [ ]:
# Predict on test
proba_list_test = rf_full.predict_proba(X_test_filtered)
y_test_proba = np.array([p[:, 1] if p.shape[1] == 2 else p[:, 0] for p in proba_list_test]).T

# Apply DAG consistency
y_test_proba_dag = enforce_dag_consistency(y_test_proba, parents_map, id_to_idx)

# Apply thresholds
y_test_pred = np.zeros_like(y_test_proba_dag, dtype=int)
for i in range(len(label_cols)):
    y_test_pred[:, i] = (y_test_proba_dag[:, i] >= optimal_thresholds_dag[i]).astype(int)

print(f"Test predictions: {y_test_pred.shape}, avg per sample: {y_test_pred.sum(axis=1).mean():.1f}")

In [ ]:
# Save submission
submission = pd.DataFrame(y_test_pred, columns=label_cols)
submission.insert(0, 'mol_id', test_df['mol_id'].values)
submission.insert(1, 'SMILES', test_df['SMILES'].values)
submission.to_parquet('../submissions/rf_stacked.parquet', index=False)
print(f"Saved submission: {submission.shape}")

In [ ]:
# Summary
print("="*60)
print("SUMMARY - Feature Stacking")
print("="*60)
print(f"Features: ECFP + MACCS + AtomPair + Torsion")
print(f"  Raw dims: {X_train.shape[1]}, After variance filter: {X_train_filtered.shape[1]}")
print(f"Baseline (threshold=0.5):       Val F1 = {f1_default:.4f}")
print(f"+ Threshold tuning:             Val F1 = {np.mean(per_class_f1):.4f}")
print(f"+ DAG + Threshold tuning:       Val F1 = {np.mean(per_class_f1_dag):.4f}")
print("="*60)
print("Compare to ECFP-only tuned:     Val F1 = 0.7012")